# ¿Por qué usar `unstructured-client` + `langchain-unstructured` en lugar del paquete completo `unstructured`?

## Introducción: El problema al que nos enfrentamos

Cuando construyes aplicaciones de IA con LangChain, uno de los primeros retos que encontrarás es: **¿Cómo extraigo texto de PDFs, documentos de Word, presentaciones de PowerPoint e imágenes en un formato que mi IA pueda entender?**

Aquí es donde entra Unstructured: es una herramienta potente que extrae texto limpio y estructurado de documentos desordenados. Pero hay un problema: instalar el paquete completo `unstructured` puede ser una pesadilla, especialmente para principiantes.

Déjame contarte lo que pasó en nuestro caso...

## La pesadilla de la instalación: Qué salió mal

Inicialmente intentamos instalar el paquete completo de Unstructured con todas las funcionalidades:

```bash
poetry add "unstructured[all-docs]"
```

**¿Resultado? ¡Un fracaso espectacular!** ❌

La instalación falló con un error sobre `llvmlite`, una biblioteca de bajo nivel que no compilaba en Python 3.13. Aquí está el porqué:

### Por qué el paquete completo es tan pesado

El paquete completo `unstructured` incluye:

1. **Modelos de aprendizaje automático** - PyTorch, TensorFlow y otras bibliotecas enormes
2. **Herramientas de procesamiento de imágenes** - OpenCV, Pillow y bibliotecas de visión artificial
3. **Motores OCR** - Tesseract y PaddleOCR para leer texto en imágenes
4. **Procesadores de PDF** - Múltiples bibliotecas de análisis de PDF
5. **Computación científica** - NumPy, SciPy y bibliotecas numéricas
6. **Requisitos de compilación** - Compiladores de C/C++ y dependencias del sistema

**¿Tamaño total de la instalación?** ¡Más de **3-5 GB** de dependencias!

Peor aún, algunos de estos paquetes (como `llvmlite`) necesitan compilarse desde el código fuente, lo que significa:
- Necesitas tener compiladores de C++ instalados
- La compatibilidad con tu versión de Python es impredecible
- La instalación puede tardar entre 15 y 30 minutos (si funciona)
- Podrías necesitar bibliotecas del sistema como `libmagic`, `poppler`, `tesseract`

## La solución moderna: Procesamiento en la nube

En lugar de ejecutar todo este procesamiento pesado en tu ordenador, hay un **enfoque mucho mejor**: enviar tus documentos a la API en la nube de Unstructured, dejar que sus servidores hagan el trabajo duro y recibir datos limpios y estructurados.

Aquí es donde entran `unstructured-client` y `langchain-unstructured`.

## Entendiendo los dos paquetes

### 1. `unstructured-client` - El cliente ligero de la API

**Qué es:** Un paquete pequeño de Python (~94 KB) que se comunica con la API en la nube de Unstructured.

**Qué hace:**
- Envía tus documentos a los servidores de Unstructured
- Gestiona la autenticación con tu clave API
- Maneja reintentos y gestión de errores
- Devuelve datos procesados y estructurados

**Qué NO incluye:**
- Sin modelos de aprendizaje automático
- Sin dependencias pesadas
- Sin requisitos de compilación
- Sin instalaciones de varios gigabytes

Piensa en ello como usar la API de Google Translate en lugar de descargar un motor de traducción completo en tu ordenador.

### 2. `langchain-unstructured` - La integración con LangChain

**Qué es:** Un paquete asociado de LangChain que conecta `unstructured-client` con los cargadores de documentos de LangChain.

**Qué hace:**
- Proporciona la clase `UnstructuredLoader` que funciona perfectamente con LangChain 1.0
- Gestiona la carga y segmentación de documentos
- Convierte la salida de Unstructured en objetos `Document` de LangChain
- Soporta tanto el modo API como el procesamiento local

**Por qué es un paquete separado:** LangChain 1.0 movió las integraciones a repositorios asociados para mantener la biblioteca principal enfocada y ligera.

## Las ventajas: Por qué este enfoque es mejor

### 1. **La instalación es instantánea** ⚡

```bash
# Forma antigua - falla tras 15 minutos
poetry add "unstructured[all-docs]"  # ❌ 5GB, errores de compilación

# Forma nueva - se instala en segundos
poetry add unstructured-client langchain-unstructured  # ✅ ~1MB en total
```

### 2. **Funciona en cualquier versión de Python** 🐍

- Sin necesidad de compilación = sin problemas de compatibilidad
- Funciona perfectamente con Python 3.10, 3.11, 3.12 **y 3.13**
- No se requieren dependencias del sistema

### 3. **Mejor calidad de procesamiento** 🎯

La API en la nube de Unstructured utiliza:
- Modelos de aprendizaje automático más potentes de los que podrías ejecutar localmente
- Mejores motores OCR
- Comprensión avanzada de documentos
- Actualizaciones periódicas sin que cambies ningún código

### 4. **Nivel gratuito generoso** 💰

- **15.000 páginas GRATIS** (¡sin fecha de caducidad!)
- Para que te hagas una idea: son unos 150 libros en PDF
- Perfecto para aprendizaje y desarrollo
- Solo pagas si necesitas más: 0,03 $/página

### 5. **Sin uso de recursos locales** 💻

- Tu ordenador no necesita ejecutar modelos de ML pesados
- Sin picos de RAM/CPU durante el procesamiento de documentos
- Funciona genial incluso en portátiles antiguos

### 6. **Totalmente compatible con LangChain 1.0** 🦜

- `langchain-unstructured` versión 1.0.1 (publicado en diciembre de 2025)
- Construido específicamente para la arquitectura de LangChain 1.0
- Mantenido por el equipo de LangChain

## Cómo usarlos: Guía paso a paso

### Paso 1: Instalación

```bash
poetry add unstructured-client langchain-unstructured
```

### Paso 2: Obtener tu clave API

1. Regístrate en [https://unstructured.io/](https://unstructured.io/)
2. Consigue tu clave API gratuita (¡15.000 páginas gratis!)
3. Añádela a tu fichero `.env`:

```bash
UNSTRUCTURED_API_KEY=tu_clave_api_aquí
```

### Paso 3: Uso básico

```python
import os
from dotenv import load_dotenv
from langchain_unstructured import UnstructuredLoader

# Cargar variables de entorno
load_dotenv()

# Crear el cargador
loader = UnstructuredLoader(
    file_path=["documento.pdf", "presentacion.pptx"],  # ¡Múltiples ficheros!
    api_key=os.getenv("UNSTRUCTURED_API_KEY"),
    partition_via_api=True,  # Usar API en la nube
    chunking_strategy="by_title",  # Segmentación inteligente para RAG
    strategy="fast"  # o "hi_res" para mejor calidad
)

# Cargar documentos
docs = loader.load()

# ¡Eso es todo! Ya tienes documentos de LangChain listos para usar
print(f"Se cargaron {len(docs)} fragmentos de documentos")
print(docs[0].page_content[:200])  # Primeros 200 caracteres
print(docs[0].metadata)  # Metadatos enriquecidos
```

### Paso 4: Usar con un pipeline RAG de LangChain

```python
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.chains import RetrievalQA

# Crear embeddings y almacenar en base de datos vectorial
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings()
)

# Crear una cadena de preguntas y respuestas
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-4"),
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# ¡Haz preguntas sobre tus documentos!
result = qa_chain({"query": "¿Cuáles son los puntos principales de este documento?"})
print(result["result"])
```

## Tipos de ficheros soportados

La API de Unstructured soporta más de **60 formatos de ficheros**, incluyendo:

**Documentos:**
- PDF, DOCX, DOC, RTF, ODT, TXT, MD, HTML

**Presentaciones:**
- PPTX, PPT, KEY

**Hojas de cálculo:**
- XLSX, XLS, CSV, TSV

**Imágenes:**
- PNG, JPG, JPEG, TIFF, BMP, HEIC

**Multimedia:**
- MP3, MP4, AVI, MOV (conversión de voz a texto)

**¡Y muchos más!**

## Funcionalidades avanzadas

### Estrategias de segmentación inteligente

```python
loader = UnstructuredLoader(
    file_path="documento_largo.pdf",
    api_key=os.getenv("UNSTRUCTURED_API_KEY"),
    partition_via_api=True,
    
    # Opciones de segmentación
    chunking_strategy="by_title",  # Segmenta por secciones del documento
    # Otras opciones: "by_page", "by_similarity", "basic"
    
    max_characters=1000,  # Tamaño máximo de fragmento
    new_after_n_chars=800,  # Intenta segmentar antes de este tamaño
    overlap=100  # Solapamiento entre fragmentos para mantener el contexto
)
```

### Múltiples estrategias de procesamiento

```python
# Estrategia rápida - procesamiento veloz
loader = UnstructuredLoader(
    file_path="documento.pdf",
    api_key=os.getenv("UNSTRUCTURED_API_KEY"),
    partition_via_api=True,
    strategy="fast"  # Ideal para documentos de texto sencillo
)

# Estrategia de alta resolución - mejor para diseños complejos
loader = UnstructuredLoader(
    file_path="diseño_complejo.pdf",
    api_key=os.getenv("UNSTRUCTURED_API_KEY"),
    partition_via_api=True,
    strategy="hi_res"  # Usa modelos de ML avanzados
)
```

### Procesamiento por lotes

```python
import glob

# Procesar todos los PDFs de un directorio
ficheros_pdf = glob.glob("./documentos/*.pdf")

loader = UnstructuredLoader(
    file_path=ficheros_pdf,  # Pasar lista de ficheros
    api_key=os.getenv("UNSTRUCTURED_API_KEY"),
    partition_via_api=True,
    chunking_strategy="by_title"
)

docs = loader.load()
print(f"Se procesaron {len(ficheros_pdf)} ficheros en {len(docs)} fragmentos")
```

## ¿Cuándo usarías el paquete local?

SÍ hay razones válidas para usar el paquete completo `unstructured` de forma local:

1. **Requisitos de privacidad** - No puedes enviar datos a APIs externas bajo ningún concepto
2. **Sin acceso a Internet** - Entornos aislados de la red
3. **Modelos personalizados** - Necesitas usar tus propios modelos entrenados
4. **Volumen muy alto** - Procesamiento de millones de páginas al mes (aunque la API podría seguir siendo más barata)

Si te encuentras en alguno de estos casos, usarías:

```bash
poetry add unstructured
```

Pero prepárate para:
- Un proceso de instalación complejo
- Posiblemente necesitar Python 3.12 en lugar de 3.13
- Instalar dependencias del sistema
- Solucionar errores de compilación

## Comparación de costes

Supongamos que estás construyendo un asistente de estudio que procesa libros de texto:

**Usando el enfoque con la API:**
- Nivel gratuito: 15.000 páginas
- Libro de texto medio: 300 páginas
- Puedes procesar: **50 libros de texto gratis**
- A partir de ahí: 0,03 $/página = 9 $ por un libro de 300 páginas

**Usando procesamiento local:**
- Tiempo de configuración: 2-4 horas (si tiene éxito)
- Requisitos del ordenador: CPU moderna, 8 GB+ de RAM
- Costes de electricidad: varían según el uso
- Tiempo de desarrollo ahorrado: ¡incalculable!

## Conclusión: El claro ganador

Para el **99 % de los casos de uso**, especialmente si eres principiante, el enfoque `unstructured-client` + `langchain-unstructured` es el claro ganador:

✅ Se instala en segundos, no en horas  
✅ Funciona de forma fiable en todos los sistemas  
✅ Mejor calidad de procesamiento  
✅ Nivel gratuito generoso  
✅ Mantenimiento cero  
✅ Totalmente compatible con LangChain 1.0  

La única contrapartida real es que necesitas conexión a Internet y estás usando una API de terceros, pero para aprendizaje y la mayoría de casos de uso en producción, esto no supone un problema.

## Lista de verificación para empezar

¿Listo para empezar? Aquí tienes tu lista de verificación:

- [ ] Instalar paquetes: `poetry add unstructured-client langchain-unstructured`
- [ ] Registrarse en [https://unstructured.io/](https://unstructured.io/)
- [ ] Añadir la clave API al fichero `.env`
- [ ] Copiar el código de uso básico de arriba
- [ ] Probar con un fichero PDF o DOCX
- [ ] ¡Construir tu aplicación RAG!

¡Feliz procesamiento de documentos! 🚀

---

**Recursos adicionales:**
- [Documentación de Unstructured](https://docs.unstructured.io/)
- [Integración de LangChain con Unstructured](https://python.langchain.com/docs/integrations/document_loaders/unstructured_file/)
- [Precios de la API de Unstructured](https://unstructured.io/pricing)